# 10 Citation Grounding Engine

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `10-citation-grounding-engine.ipynb`

In [1]:
# Start coding here
# ==================================================
# Notebook 10
# Citation Grounding Engine
# ==================================================

import pandas as pd
import json

In [2]:
parents_df = pd.read_csv("artifacts/parent_chunks.csv")

metadata_df = pd.read_csv("artifacts/document_metadata.csv")

In [3]:
parent_metadata = parents_df.merge(
    metadata_df[["document_id", "filename", "department"]], on="document_id", how="left"
)

parent_metadata.head()

,parent_id,document_id,content,filename,department
0,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,AI Platform Roadmap\nSearch Infrastructure\nRe...,AI_Roadmap.txt,engineering
1,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,er deployment planned.\nEnterprise RAG platfor...,AI_Roadmap.txt,engineering
2,9cb41e2d-b979-4c89-a091-01db42c3748e,b638264b-b788-442e-a09d-19176466fba8,Annual Financial Report 2026\nRevenue Performa...,Annual_Report_2026.txt,finance
3,7cdee370-fba0-4bc7-9fe1-a39230162073,b638264b-b788-442e-a09d-19176466fba8,oud division grew by 35%.\nAI products generat...,Annual_Report_2026.txt,finance
4,fd7bb5e1-9ec6-470a-8726-bbdc04fcc6dc,31516a79-2cf5-4080-ac66-66b338d6ebff,Employee Benefits 2026\nHealth Insurance\nDent...,Benefits_2026.txt,hr


In [4]:
citation_schema = {
    "citation_id": "",
    "filename": "",
    "department": "",
    "document_id": "",
    "parent_id": "",
    "evidence": "",
}

citation_schema

{'citation_id': '',
 'filename': '',
 'department': '',
 'document_id': '',
 'parent_id': '',
 'evidence': ''}

In [5]:
import uuid


def build_citation(row):

    return {
        "citation_id": str(uuid.uuid4()),
        "filename": row["filename"],
        "department": row["department"],
        "document_id": row["document_id"],
        "parent_id": row["parent_id"],
        "evidence": row["content"],
    }

In [6]:
sample = parent_metadata.iloc[0]

build_citation(sample)

{'citation_id': '30ed0431-08e8-47b0-8df6-78703bc2c7ee',
 'filename': 'AI_Roadmap.txt',
 'department': 'engineering',
 'document_id': '1fe99400-06e1-4964-956b-3861687e7da4',
 'parent_id': '7d09d047-2858-43c4-bf8b-d40519129087',
 'evidence': 'AI Platform Roadmap\nSearch Infrastructure\nRetrieval Systems\nDeployment Roadmap\nVector search infrastructure rollout.\nSearch latency improved significantly.\nNew indexing pipeline deployed.\nHybrid retrieval implementation.\nSemantic search quality improved.\nKnowledge base coverage expanded.\nCross encod'}

In [7]:
citation_store = {}

for _, row in parent_metadata.iterrows():

    citation_store[row["parent_id"]] = build_citation(row)

In [8]:
def get_citation(parent_id):

    return citation_store.get(parent_id)

In [9]:
first_parent = parent_metadata.iloc[0]["parent_id"]

get_citation(first_parent)

{'citation_id': 'f1504e56-4aec-4fab-a41a-8fa78f621f31',
 'filename': 'AI_Roadmap.txt',
 'department': 'engineering',
 'document_id': '1fe99400-06e1-4964-956b-3861687e7da4',
 'parent_id': '7d09d047-2858-43c4-bf8b-d40519129087',
 'evidence': 'AI Platform Roadmap\nSearch Infrastructure\nRetrieval Systems\nDeployment Roadmap\nVector search infrastructure rollout.\nSearch latency improved significantly.\nNew indexing pipeline deployed.\nHybrid retrieval implementation.\nSemantic search quality improved.\nKnowledge base coverage expanded.\nCross encod'}

In [10]:
retrieved_docs = [{"parent_id": parent_metadata.iloc[0]["parent_id"], "score": 0.92}]

In [11]:
def attach_citations(retrieved_docs):

    results = []

    for doc in retrieved_docs:

        citation = get_citation(doc["parent_id"])

        results.append({"score": doc["score"], "citation": citation})

    return results

In [12]:
grounded_docs = attach_citations(retrieved_docs)

grounded_docs

[{'score': 0.92,
  'citation': {'citation_id': 'f1504e56-4aec-4fab-a41a-8fa78f621f31',
   'filename': 'AI_Roadmap.txt',
   'department': 'engineering',
   'document_id': '1fe99400-06e1-4964-956b-3861687e7da4',
   'parent_id': '7d09d047-2858-43c4-bf8b-d40519129087',
   'evidence': 'AI Platform Roadmap\nSearch Infrastructure\nRetrieval Systems\nDeployment Roadmap\nVector search infrastructure rollout.\nSearch latency improved significantly.\nNew indexing pipeline deployed.\nHybrid retrieval implementation.\nSemantic search quality improved.\nKnowledge base coverage expanded.\nCross encod'}}]

In [13]:
def build_grounded_context(documents):

    context = ""

    for doc in documents:

        citation = doc["citation"]

        context += f"""

SOURCE:
{citation['filename']}

DEPARTMENT:
{citation['department']}

CONTENT:
{citation['evidence']}

==================================================

"""

    return context

In [14]:
context = build_grounded_context(grounded_docs)

print(context)



SOURCE:
AI_Roadmap.txt

DEPARTMENT:
engineering

CONTENT:
AI Platform Roadmap
Search Infrastructure
Retrieval Systems
Deployment Roadmap
Vector search infrastructure rollout.
Search latency improved significantly.
New indexing pipeline deployed.
Hybrid retrieval implementation.
Semantic search quality improved.
Knowledge base coverage expanded.
Cross encod





In [15]:
def create_grounded_prompt(query, context):

    prompt = f"""
You are an enterprise AI assistant.

Answer ONLY using the provided context.

If information is missing,
say:

'I could not find that information.'

CONTEXT:

{context}

QUESTION:

{query}

ANSWER:
"""

    return prompt

In [16]:
query = "What was revenue?"

prompt = create_grounded_prompt(query, context)

print(prompt)


You are an enterprise AI assistant.

Answer ONLY using the provided context.

If information is missing,
say:

'I could not find that information.'

CONTEXT:



SOURCE:
AI_Roadmap.txt

DEPARTMENT:
engineering

CONTENT:
AI Platform Roadmap
Search Infrastructure
Retrieval Systems
Deployment Roadmap
Vector search infrastructure rollout.
Search latency improved significantly.
New indexing pipeline deployed.
Hybrid retrieval implementation.
Semantic search quality improved.
Knowledge base coverage expanded.
Cross encod




QUESTION:

What was revenue?

ANSWER:



In [17]:
def format_citations(grounded_docs):

    citations = []

    for doc in grounded_docs:

        citation = doc["citation"]

        citations.append(
            {"filename": citation["filename"], "department": citation["department"]}
        )

    return citations

In [18]:
format_citations(grounded_docs)

[{'filename': 'AI_Roadmap.txt', 'department': 'engineering'}]

In [19]:
{
    "document": "Annual_Report_2026.pdf",
    "page": 12,
    "paragraph": 4,
    "chunk_id": "xyz",
    "confidence": 0.92,
}

{'document': 'Annual_Report_2026.pdf',
 'page': 12,
 'paragraph': 4,
 'chunk_id': 'xyz',
 'confidence': 0.92}

In [20]:
enhanced_citation = {
    "document": "",
    "page": "",
    "section": "",
    "paragraph": "",
    "chunk_id": "",
    "retrieval_score": "",
    "rerank_score": "",
}

In [21]:
with open("artifacts/citation_store.json", "w") as file:

    json.dump(citation_store, file, indent=4)